# Patient-wise Dataset Splitting

This notebook shows how to use `group_by_patient()` and `split(by_patient=True)` on the **BUSI** dataset.

## The Problem: Data Leakage

Standard random splitting shuffles individual resources. If a patient has **multiple scans** (e.g. a baseline CT and a follow-up CT), both scans can end up in different splits (one in `train`, one in `test`). Because the images share patient anatomy, the model effectively sees the test patient during training, inflating metrics.

**Patient-wise splitting** fixes this by shuffling *patients* instead of *resources*: every scan belonging to a patient lands in the same split.

```
Resource-wise (risky)       Patient-wise (safe)
──────────────────────      ──────────────────────
train: scan_A1, scan_B1     train: patient_A (scan_A1, scan_A2)
test:  scan_A2, scan_B2     test:  patient_B (scan_B1, scan_B2)
       ↑ same patient!             ↑ clean boundary
```

## What You'll Learn

1. How `patient_id` is exposed on resources
2. `group_by_patient()` — organise a dataset by patient for inspection or custom splits
3. `split(by_patient=True)` — an option for the split
4. How to persist patient-wise splits to the server and reload them reproducibly
5. How to handle resources without a `patient_id`

## 1. Setup

This notebook assumes the BUSI project has already been created and populated.
If not, follow `segmentation_2d_trainer_BUSI_tutorial.ipynb` first.

In [2]:
from datamint import Api
from datamint.dataset import ImageDataset

PROJECT_NAME = "TransUNet_BUSI_Tutorial"

api = Api()
project = api.projects.get_by_name(PROJECT_NAME)

dataset = ImageDataset(project=project, include_unannotated=True)
print(f"Loaded {len(dataset)} resources")

Loaded 780 resources


## 2. Inspecting `patient_id`

Let's check the first few resources. For BUSI (PNG files), patient_id is None.

In [3]:
for r in dataset.resources[:5]:
    print(f"{r.filename:40s}  patient_id={r.patient_id!r}")

benign (111).png                          patient_id=None
benign (100).png                          patient_id=None
benign (110).png                          patient_id=None
benign (109).png                          patient_id=None
benign (101).png                          patient_id=None


In [4]:
with_id    = sum(1 for r in dataset.resources if dataset._get_patient_id(r) is not None)
without_id = len(dataset) - with_id
print(f"With patient_id   : {with_id}")
print(f"Without patient_id: {without_id}")

With patient_id   : 0
Without patient_id: 780


## 3. `group_by_patient()`

`group_by_patient()` returns a `dict` mapping each patient ID to a sub-dataset containing only that patient's resources. 

### Handling `patient_id=None`

In this case, every resource has `patient_id=None`. The `none_patient_id_strategy` parameter controls what to do:

| Strategy | Behaviour |
|---|---|
| `'individual'` (default) | Each `None`-patient resource becomes its own group (no leakage possible) |
| `'group'` | All grouped together under key `None` |
| `'skip'` | Excluded from the output entirely |
| `'error'` | Raises `ValueError` immediately |

For BUSI, `'individual'` is correct: each image is a distinct case.

In [5]:
groups = dataset.group_by_patient(none_patient_id_strategy='individual')

sizes = [len(g) for g in groups.values()]
print(f"Patient groups : {len(groups)}")
print(f"Total resources: {sum(sizes)}")
print(f"Per-group size : min={min(sizes)}, max={max(sizes)}, mean={sum(sizes)/len(sizes):.1f}")

Patient groups : 780
Total resources: 780
Per-group size : min=1, max=1, mean=1.0


## 4. `split(by_patient=True)` — Compute Patient-wise Splits Locally

When passing by_patient=True, the split shuffles patients instead of individual resources. Internally it shuffles **patients**, assigns patient buckets to splits by ratio, then collects all resources from each bucket.

In [6]:
parts = dataset.split(
    train=0.7,
    val=0.15,
    test=0.15,
    by_patient=True,
    none_patient_id_strategy='individual',  # correct for PNG/BUSI
    seed=42,
)

for name, ds in parts.items():
    print(f"{name:6s}: {len(ds):4d} resources")

train :  546 resources
val   :  117 resources
test  :  117 resources


In [7]:
# Verify no patient appears in more than one split
train_pids = {dataset._get_patient_id(r) or r.id for r in parts['train'].resources}
val_pids   = {dataset._get_patient_id(r) or r.id for r in parts['val'].resources}
test_pids  = {dataset._get_patient_id(r) or r.id for r in parts['test'].resources}

print("train ∩ val  :", len(train_pids & val_pids),  "patients in common")
print("train ∩ test :", len(train_pids & test_pids), "patients in common")
print("val   ∩ test :", len(val_pids   & test_pids), "patients in common")

assert not (train_pids & val_pids)
assert not (train_pids & test_pids)
assert not (val_pids   & test_pids)
print("✓ No patient leakage across splits.")

train ∩ val  : 0 patients in common
train ∩ test : 0 patients in common
val   ∩ test : 0 patients in common
✓ No patient leakage across splits.


## 5. Persist Splits to the Server

The `parts` dict computed above lives only in memory. To make the split **reproducible across sessions and shareable with the team**, write the assignments back to the project using `api.projects.assign_splits()`.

Once persisted, anyone can reload exactly the same split (without re-running the patient split logic) by calling `dataset.split()` with no arguments (project-backed datasets prefer the server assignments automatically).

In [8]:
for split_name, split_ds in parts.items():
    api.projects.assign_splits(project, split_ds.resources, split_name=split_name)
    print(f"Assigned {len(split_ds):4d} resources → '{split_name}'")

print("\nPatient-wise split assignments saved to the project.")

Assigned  546 resources → 'train'
Assigned  117 resources → 'val'
Assigned  117 resources → 'test'

Patient-wise split assignments saved to the project.


## 6. Reload Splits from the Server

After persisting, the splits are loaded back just like any project-scoped split, no `by_patient`, no ratios. The `split_as_of_timestamp` returned on each sub-dataset can be stored and replayed later to recover the exact same snapshot.

In [9]:
# Reload the dataset fresh (simulates a new session)
dataset_fresh = ImageDataset(project=project, include_unannotated=True)

# No by_patient, no ratios — reads from project API
reloaded = dataset_fresh.split()

for name, ds in reloaded.items():
    print(f"{name:6s}: {len(ds):4d} resources  "
          f"split_source={ds.split_source!r}  "
          f"as_of={ds.split_as_of_timestamp}")

val   :  117 resources  split_source='project_api'  as_of=2026-06-11T12:00:09.030466Z
train :  546 resources  split_source='project_api'  as_of=2026-06-11T12:00:09.030466Z
test  :  117 resources  split_source='project_api'  as_of=2026-06-11T12:00:09.030466Z


In [10]:
# Replay the exact same snapshot at any later point
snapshot_ts = reloaded['train'].split_as_of_timestamp

replayed = dataset_fresh.split(as_of_timestamp=snapshot_ts)
print(f"Replaying split snapshot from {snapshot_ts}")
print({name: len(ds) for name, ds in replayed.items()})

Replaying split snapshot from 2026-06-11T12:00:09.030466Z
{'val': 117, 'train': 546, 'test': 117}
